In [2]:
from pathlib import Path
import sys
import json
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
)
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

In [3]:
CURRENT_DIR = Path.cwd()
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [4]:
from src.features.preprocessing import build_preprocessor

In [5]:
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "telco_churn_clean.csv"
)

MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "split_manifest.csv"
)

COMPARISON_PATH = (
    PROJECT_ROOT
    / "reports"
    / "model_comparison_cv.csv"
)

df = pd.read_csv(DATA_PATH)
manifest = pd.read_csv(MANIFEST_PATH)
comparison = pd.read_csv(COMPARISON_PATH)

print("Dataset:", df.shape)
print("Manifest:", manifest.shape)
print("Model comparison:", comparison.shape)

comparison

Dataset: (7043, 21)
Manifest: (7043, 3)
Model comparison: (6, 11)


,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC-AUC,Average Precision,CV Time (sec),F1 vs Baseline,Recall vs Baseline
0,XGBoost Weighted,0.759494,0.763672,0.532548,0.772575,0.630355,0.843803,0.665835,2.275842,0.030807,0.222742
1,XGBoost,0.803160,0.717119,0.660124,0.533779,0.590130,0.845432,0.665707,3.464691,-0.009417,-0.016054
2,Logistic Regression,0.805290,0.723698,0.660232,0.549833,0.599548,0.845775,0.661096,4.572300,0.000000,0.000000
3,Logistic Regression Balanced,0.750089,0.766458,0.518947,0.801338,0.629912,0.845786,0.659441,4.735668,0.030364,0.251505
4,Random Forest,0.789849,0.695881,0.633099,0.495652,0.555641,0.823973,0.619013,4.445053,-0.043907,-0.054181
5,Random Forest Balanced,0.775826,0.735049,0.567987,0.648161,0.605365,0.826224,0.614609,4.455161,0.005818,0.098328


In [6]:
data = df.merge(
    manifest[["customerID", "split"]],
    on="customerID",
    how="left",
    validate="one_to_one"
)

train_data = data[
    data["split"] == "train"
].copy()

X_train = train_data.drop(
    columns=[
        "customerID",
        "Churn",
        "split",
    ]
)

y_train = train_data["Churn"].map({
    "No": 0,
    "Yes": 1,
})

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train: (5634, 19)
y_train: (5634,)


In [7]:
#Imbalance ratio
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())

scale_pos_weight = float(
    negative_count / positive_count
)

print("Negative:", negative_count)
print("Positive:", positive_count)
print(
    "Negative / Positive:",
    round(scale_pos_weight, 4)
)

Negative: 4139
Positive: 1495
Negative / Positive: 2.7686


### Group weighted/unweighted versions of the same model

In [8]:
def get_model_family(model_name):

    if "Logistic Regression" in model_name:
        return "Logistic Regression"

    if "Random Forest" in model_name:
        return "Random Forest"

    if "XGBoost" in model_name:
        return "XGBoost"

    raise ValueError(
        f"Unknown model: {model_name}"
    )


comparison["Family"] = (
    comparison["Model"]
    .apply(get_model_family)
)

In [9]:
comparison[
    [
        "Model",
        "Family",
        "Average Precision",
        "F1",
        "Recall",
        "ROC-AUC",
    ]
].sort_values(
    "Average Precision",
    ascending=False
).round(4)

,Model,Family,Average Precision,F1,Recall,ROC-AUC
0,XGBoost Weighted,XGBoost,0.6658,0.6304,0.7726,0.8438
1,XGBoost,XGBoost,0.6657,0.5901,0.5338,0.8454
2,Logistic Regression,Logistic Regression,0.6611,0.5995,0.5498,0.8458
3,Logistic Regression Balanced,Logistic Regression,0.6594,0.6299,0.8013,0.8458
4,Random Forest,Random Forest,0.6190,0.5556,0.4957,0.8240
5,Random Forest Balanced,Random Forest,0.6146,0.6054,0.6482,0.8262


#### Bring out the best version of each family.

In [10]:
family_best = (
    comparison
    .sort_values(
        "Average Precision",
        ascending=False
    )
    .drop_duplicates(
        subset="Family",
        keep="first"
    )
    .reset_index(drop=True)
)

family_best[
    [
        "Family",
        "Model",
        "Average Precision",
        "F1",
        "Recall",
        "ROC-AUC",
    ]
].round(4)

,Family,Model,Average Precision,F1,Recall,ROC-AUC
0,XGBoost,XGBoost Weighted,0.6658,0.6304,0.7726,0.8438
1,Logistic Regression,Logistic Regression,0.6611,0.5995,0.5498,0.8458
2,Random Forest,Random Forest,0.6190,0.5556,0.4957,0.8240


In [11]:
#Top 2 model families select
top_two = family_best.head(2)

top_families = (
    top_two["Family"]
    .tolist()
)

print("Models selected for tuning:")

for i, model in enumerate(
    top_families,
    start=1
):
    print(f"{i}. {model}")

Models selected for tuning:
1. XGBoost
2. Logistic Regression


In [12]:
#Cross-validation strategy
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [13]:
#Metrics
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy":
        "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision":
        "average_precision",
}

#### Model builder function

In [14]:
def build_model_pipeline(family):

    if family == "Logistic Regression":

        classifier = LogisticRegression(
            max_iter=3000,
            solver="liblinear",
            random_state=42,
        )

    elif family == "Random Forest":

        classifier = RandomForestClassifier(
            random_state=42,
            n_jobs=1,
        )

    elif family == "XGBoost":

        classifier = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,
            n_jobs=1,
        )

    else:
        raise ValueError(
            f"Unknown family: {family}"
        )

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                build_preprocessor()
            ),
            (
                "classifier",
                classifier
            ),
        ]
    )

    return pipeline

In [15]:
#Logistic Regression search space
logistic_params = {
    "classifier__C": [
        0.01,
        0.03,
        0.1,
        0.3,
        1.0,
        3.0,
        10.0,
        30.0,
    ],

    "classifier__penalty": [
        "l1",
        "l2",
    ],

    "classifier__class_weight": [
        None,
        "balanced",
    ],
}

In [16]:
#Random Forest search space
random_forest_params = {
    "classifier__n_estimators": [
        200,
        300,
        500,
        700,
    ],

    "classifier__max_depth": [
        None,
        5,
        8,
        12,
        16,
    ],

    "classifier__min_samples_split": [
        2,
        5,
        10,
    ],

    "classifier__min_samples_leaf": [
        1,
        2,
        4,
        8,
    ],

    "classifier__max_features": [
        "sqrt",
        "log2",
        None,
    ],

    "classifier__class_weight": [
        None,
        "balanced",
        "balanced_subsample",
    ],
}

In [18]:
#XGBoost search space
xgb_weight_values = [
    1.0,
    round(scale_pos_weight * 0.75, 4),
    round(scale_pos_weight, 4),
    round(scale_pos_weight * 1.25, 4),
    round(scale_pos_weight * 1.50, 4),
]

xgb_params = {
    "classifier__n_estimators": [
        150,
        250,
        350,
        500,
    ],

    "classifier__learning_rate": [
        0.01,
        0.03,
        0.05,
        0.1,
    ],

    "classifier__max_depth": [
        2,
        3,
        4,
        5,
        6,
    ],

    "classifier__min_child_weight": [
        1,
        3,
        5,
    ],

    "classifier__subsample": [
        0.7,
        0.8,
        0.9,
        1.0,
    ],

    "classifier__colsample_bytree": [
        0.7,
        0.8,
        0.9,
        1.0,
    ],

    "classifier__reg_alpha": [
        0.0,
        0.1,
        0.5,
    ],

    "classifier__reg_lambda": [
        1.0,
        2.0,
        5.0,
    ],

    "classifier__scale_pos_weight":
        xgb_weight_values,
}

#### Search configuration mapping


In [19]:
parameter_spaces = {
    "Logistic Regression":
        logistic_params,

    "Random Forest":
        random_forest_params,

    "XGBoost":
        xgb_params,
}

In [20]:
#Search iterations:
search_iterations = {
    "Logistic Regression": 20,
    "Random Forest": 20,
    "XGBoost": 25,
}

#### Main hyperparameter tuning

In [21]:
tuning_results = []
best_estimators = {}
best_params = {}

for family in top_families:

    print("=" * 60)
    print(f"Tuning: {family}")
    print("=" * 60)

    pipeline = build_model_pipeline(
        family
    )

    search = RandomizedSearchCV(
        estimator=pipeline,

        param_distributions=
            parameter_spaces[family],

        n_iter=
            search_iterations[family],

        scoring=scoring,

        refit="average_precision",

        cv=cv,

        n_jobs=-1,

        random_state=42,

        verbose=1,

        return_train_score=False,
    )

    start_time = time.time()

    search.fit(
        X_train,
        y_train
    )

    elapsed = (
        time.time() - start_time
    )

    best_index = search.best_index_

    result = {
        "Model Family": family,

        "Average Precision":
            search.cv_results_[
                "mean_test_average_precision"
            ][best_index],

        "ROC-AUC":
            search.cv_results_[
                "mean_test_roc_auc"
            ][best_index],

        "F1":
            search.cv_results_[
                "mean_test_f1"
            ][best_index],

        "Recall":
            search.cv_results_[
                "mean_test_recall"
            ][best_index],

        "Precision":
            search.cv_results_[
                "mean_test_precision"
            ][best_index],

        "Balanced Accuracy":
            search.cv_results_[
                "mean_test_balanced_accuracy"
            ][best_index],

        "Accuracy":
            search.cv_results_[
                "mean_test_accuracy"
            ][best_index],

        "Tuning Time (sec)":
            elapsed,
    }

    tuning_results.append(
        result
    )

    best_estimators[family] = (
        search.best_estimator_
    )

    best_params[family] = (
        search.best_params_
    )

    print(
        "\nBest Average Precision:",
        round(search.best_score_, 4)
    )

    print("\nBest parameters:")

    for key, value in (
        search.best_params_.items()
    ):
        print(
            f"{key}: {value}"
        )

    # Save top search results
    search_results = pd.DataFrame(
        search.cv_results_
    )

    search_results = (
        search_results
        .sort_values(
            "rank_test_average_precision"
        )
        .head(10)
    )

    safe_name = (
        family
        .lower()
        .replace(" ", "_")
    )

    search_results.to_csv(
        PROJECT_ROOT
        / "reports"
        / f"tuning_{safe_name}.csv",
        index=False
    )

    print("\nFinished.\n")

Tuning: XGBoost
Fitting 5 folds for each of 25 candidates, totalling 125 fits

Best Average Precision: 0.673

Best parameters:
classifier__subsample: 0.8
classifier__scale_pos_weight: 2.0764
classifier__reg_lambda: 1.0
classifier__reg_alpha: 0.5
classifier__n_estimators: 150
classifier__min_child_weight: 1
classifier__max_depth: 2
classifier__learning_rate: 0.05
classifier__colsample_bytree: 0.8

Finished.

Tuning: Logistic Regression
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best Average Precision: 0.6612

Best parameters:
classifier__penalty: l1
classifier__class_weight: None
classifier__C: 1.0

Finished.



c:\AI_Projects\customer-churn-ml-system\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\AI_Projects\customer-churn-ml-system\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


#### Tuned models comparison


In [22]:
tuning_summary = pd.DataFrame(
    tuning_results
)

tuning_summary = (
    tuning_summary
    .sort_values(
        by=[
            "Average Precision",
            "F1"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

tuning_summary.round(4)

,Model Family,Average Precision,ROC-AUC,F1,Recall,Precision,Balanced Accuracy,Accuracy,Tuning Time (sec)
0,XGBoost,0.6730,0.8504,0.6370,0.7398,0.5594,0.7646,0.7762,41.7522
1,Logistic Regression,0.6612,0.8458,0.6012,0.5518,0.6612,0.7247,0.8058,10.9151


### Percentage table

In [23]:
display_summary = (
    tuning_summary.copy()
)

metric_columns = [
    "Average Precision",
    "ROC-AUC",
    "F1",
    "Recall",
    "Precision",
    "Balanced Accuracy",
    "Accuracy",
]

for column in metric_columns:
    display_summary[column] = (
        display_summary[column]
        .mul(100)
        .round(2)
    )

display_summary

,Model Family,Average Precision,ROC-AUC,F1,Recall,Precision,Balanced Accuracy,Accuracy,Tuning Time (sec)
0,XGBoost,67.30,85.04,63.70,73.98,55.94,76.46,77.62,41.752228
1,Logistic Regression,66.12,84.58,60.12,55.18,66.12,72.47,80.58,10.915087


In [24]:
# Best parameters show
for family in top_families:

    print("\n" + "=" * 50)
    print(f"BEST PARAMETERS: {family}")
    print("=" * 50)

    for parameter, value in (
        best_params[family].items()
    ):
        print(
            f"{parameter}: {value}"
        )


BEST PARAMETERS: XGBoost
classifier__subsample: 0.8
classifier__scale_pos_weight: 2.0764
classifier__reg_lambda: 1.0
classifier__reg_alpha: 0.5
classifier__n_estimators: 150
classifier__min_child_weight: 1
classifier__max_depth: 2
classifier__learning_rate: 0.05
classifier__colsample_bytree: 0.8

BEST PARAMETERS: Logistic Regression
classifier__penalty: l1
classifier__class_weight: None
classifier__C: 1.0


### Tuned vs untuned comparison

In [25]:
comparison_rows = []

for _, tuned_row in (
    tuning_summary.iterrows()
):

    family = tuned_row[
        "Model Family"
    ]

    old_row = (
        family_best[
            family_best["Family"]
            == family
        ]
        .iloc[0]
    )

    comparison_rows.append({
        "Family": family,

        "Before AP":
            old_row[
                "Average Precision"
            ],

        "After AP":
            tuned_row[
                "Average Precision"
            ],

        "AP Improvement":
            tuned_row[
                "Average Precision"
            ]
            -
            old_row[
                "Average Precision"
            ],

        "Before F1":
            old_row["F1"],

        "After F1":
            tuned_row["F1"],

        "F1 Improvement":
            tuned_row["F1"]
            -
            old_row["F1"],
    })

improvement_df = pd.DataFrame(
    comparison_rows
)

improvement_df.round(4)

,Family,Before AP,After AP,AP Improvement,Before F1,After F1,F1 Improvement
0,XGBoost,0.6658,0.6730,0.0071,0.6304,0.6370,0.0066
1,Logistic Regression,0.6611,0.6612,0.0001,0.5995,0.6012,0.0016


In [26]:
# ONE model select
selected_row = (
    tuning_summary.iloc[0]
)

selected_family = (
    selected_row["Model Family"]
)

selected_model = (
    best_estimators[
        selected_family
    ]
)

selected_params = (
    best_params[
        selected_family
    ]
)

print(
    "SELECTED MODEL:",
    selected_family
)

print(
    "CV Average Precision:",
    round(
        selected_row[
            "Average Precision"
        ],
        4
    )
)

print(
    "CV F1:",
    round(
        selected_row["F1"],
        4
    )
)

print(
    "CV Recall:",
    round(
        selected_row["Recall"],
        4
    )
)

print(
    "CV ROC-AUC:",
    round(
        selected_row["ROC-AUC"],
        4
    )
)

SELECTED MODEL: XGBoost
CV Average Precision: 0.673
CV F1: 0.637
CV Recall: 0.7398
CV ROC-AUC: 0.8504


#### Tuning summary save

In [27]:
TUNING_SUMMARY_PATH = (
    PROJECT_ROOT
    / "reports"
    / "hyperparameter_tuning_summary.csv"
)

tuning_summary.to_csv(
    TUNING_SUMMARY_PATH,
    index=False
)

print(
    "Saved:",
    TUNING_SUMMARY_PATH
)

Saved: c:\AI_Projects\customer-churn-ml-system\reports\hyperparameter_tuning_summary.csv


### Best parameters JSON save

In [28]:
BEST_PARAMS_PATH = (
    PROJECT_ROOT
    / "reports"
    / "best_tuning_parameters.json"
)

with open(
    BEST_PARAMS_PATH,
    "w"
) as file:

    json.dump(
        best_params,
        file,
        indent=4
    )

print(
    "Saved:",
    BEST_PARAMS_PATH
)

Saved: c:\AI_Projects\customer-churn-ml-system\reports\best_tuning_parameters.json


In [29]:
# Selected candidate save
SELECTED_PATH = (
    PROJECT_ROOT
    / "reports"
    / "selected_model.json"
)

selected_information = {
    "model_family":
        selected_family,

    "selection_metric":
        "Average Precision",

    "cv_average_precision":
        float(
            selected_row[
                "Average Precision"
            ]
        ),

    "cv_f1":
        float(
            selected_row["F1"]
        ),

    "cv_recall":
        float(
            selected_row["Recall"]
        ),

    "cv_roc_auc":
        float(
            selected_row["ROC-AUC"]
        ),

    "best_parameters":
        selected_params,
}

with open(
    SELECTED_PATH,
    "w"
) as file:

    json.dump(
        selected_information,
        file,
        indent=4
    )

print(
    "Selected model info saved."
)

Selected model info saved.
